**RAG Debugging**

In [18]:
import os
import time
from dotenv import load_dotenv
import os
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()

True

In [19]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")
gemini_llm = genai.Client(api_key=geminiapikey)


In [20]:
# Embeddings model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2943.21it/s]


In [21]:
file_path = "yarvalley.txt"
loader = TextLoader(file_path)
text_file = loader.load()

In [25]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n## ",   # Major sections first
        "\n### ",  # Subsections
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

split_document = splitter.split_documents(text_file)


In [26]:
client = QdrantClient(
       url=QDRANT_ENDPOINT,
       api_key=QDRANT_API_KEY

)

In [29]:
collections = client.get_collections().collections


#Any go through list and stop at true(meet your condition)
collection_exist = any(
    collection.name == "ragval"
    for collection in collections
)

if not collection_exist:
    vectorestore = QdrantVectorStore.from_documents(
        documents=split_document,
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings,
        collection_name="ragval"
    )

else:
    vectorestore = QdrantVectorStore.from_existing_collection(
        collection_name="ragval",
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings
    )

    print("Collection already exists. Using existing collection.")



Collection already exists. Using existing collection.
